# 01 — Exploratory Data Analysis

**Goal:** Understand the structure of the VIX time series before modeling — level, distribution, memory, and the market regimes that dominate the data.

All plotting logic lives in `src/vix_forecasting/analysis/eda.py`. This notebook calls it and adds written interpretation.

In [ ]:
import sys
from pathlib import Path
import yaml

sys.path.insert(0, str(Path('..').resolve()))

from src.vix_forecasting.data.acquisition import download_vix, validate_raw, save_raw
from src.vix_forecasting.data.preprocessing import load_raw, build_series
from src.vix_forecasting.analysis import eda

%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 120

with open('../config/config.yaml') as f:
    cfg = yaml.safe_load(f)

FIGURES = Path('../figures')
RAW_PATH = Path('../data/raw/vix_raw.csv')

In [ ]:
df = load_raw(RAW_PATH)
vix = build_series(df)

print(f"Series length : {len(vix):,} trading days")
print(f"Date range    : {vix.index[0].date()} to {vix.index[-1].date()}")
print(f"Min / Max     : {vix.min():.2f} / {vix.max():.2f}")
print(f"Mean / Median : {vix.mean():.2f} / {vix.median():.2f}")
print(f"Std dev       : {vix.std():.2f}")
vix.head(3)

## Full History

Shaded regions mark the four major volatility regimes in the sample.

In [ ]:
fig = eda.plot_full_history(vix, save_path=FIGURES / 'vix_full_history.png')
plt.show()

**Commentary:**

Several structural features are immediately visible:

- **Mean-reversion tendency**: VIX repeatedly spikes then decays back toward a long-run level (~20). This is the defining characteristic of volatility — it clusters but does not trend indefinitely.
- **Spike asymmetry**: Rises are sharp and abrupt; decays are slow. This creates a right-skewed level distribution and makes the *onset* of a spike the hardest forecasting problem.
- **Regime clustering**: The four shaded regimes account for the majority of extreme readings. The GFC peak (~89) and COVID peak (~83) are far outside the normal operating range; any model evaluated without attention to these periods will be misleadingly optimistic.
- **Post-2012 low-volatility regime**: The period 2012–2017 is unusually quiet and prolonged. Models trained primarily on this window would systematically underforecast tail events.

The 9/11 closure gap (Sep 10–17, 2001) is visible as a brief flat stretch and is a known market event, not a data error.

## Distribution

In [ ]:
fig = eda.plot_distribution(vix, save_path=FIGURES / 'vix_distribution.png')
plt.show()

print(f"Skewness : {vix.skew():.2f}")
print(f"Kurtosis : {vix.kurtosis():.2f}  (excess, normal=0)")
for p in [10, 25, 50, 75, 90, 95, 99]:
    print(f"  p{p:2d}: {vix.quantile(p/100):.1f}")

**Commentary:**

- The distribution is **heavily right-skewed** (positive skewness) with **excess kurtosis** — fat tails driven by the crisis spikes. A Gaussian assumption is inappropriate for VIX levels.
- The bulk of the mass sits between ~10 and ~30. The region above 40 is rare but consequential — those are exactly the periods where volatility forecasts matter most for risk management.
- The practical implication for evaluation: RMSE and MAE will be dominated by spike episodes. We need to report directional accuracy separately to understand behavior in normal regimes vs. spike onsets.

## Autocorrelation (ACF / PACF)

In [ ]:
fig = eda.plot_acf_pacf(vix, lags=40, save_path=FIGURES / 'vix_acf_pacf.png')
plt.show()

**Commentary:**

- **ACF**: Autocorrelations decay very slowly and remain significant well beyond lag 40. This pattern is characteristic of an **integrated or near-integrated process** — the series has long memory and does not return to a fixed mean quickly at short lags.
- **PACF**: The partial autocorrelation is very large at lag 1 and cuts off sharply (with small but possibly significant values at lags 2–3). This profile is consistent with an **AR(1) or AR(2) process on the level series**, or equivalently, near-unit-root behavior.
- **Implication for ARIMA order**: The slow ACF decay strongly suggests differencing may be warranted. If ADF/KPSS testing in Phase 3 confirms non-stationarity, we'll work on the differenced series. The PACF suggests low AR order (p ∈ {1, 2}) after appropriate differencing.
- **Seasonality**: No obvious periodic spike at lag 5 (weekly) is visible in the level series. We'll confirm with a formal test in Phase 5 when we fit ARIMA/SARIMA.